<a href="https://www.kaggle.com/code/muneeburrehman98/deimv2-finetune-anime?scriptVersionId=318596661" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
!git clone https://github.com/Intellindust-AI-Lab/DEIMv2.git

In [ ]:
!pip install -q faster-coco-eval gdown onnx onnxsim onnxscript calflops tensorboard
!pip install -q -r DEIMv2/requirements.txt

In [ ]:
!hf download Intellindust/DEIMv2_DINOv3_L_COCO --local-dir ./ckpts

In [ ]:
import torch
from safetensors.torch import load_file

weights = load_file("ckpts/model.safetensors")

checkpoint = {
    "model": weights
}

torch.save(checkpoint, "deimv2.pth")

print("Successfully converted model.safetensors to deimv2.pth")

In [ ]:
!kaggle datasets download muneeburrehman98/danbooru-annotated-images

In [ ]:
!unzip -q danbooru-annotated-images.zip

In [ ]:
import json
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

root_dir = Path("dataset")
source_images_dir = root_dir / "images"  
target_dir = root_dir 
seed = 42

master_json = root_dir / "annotations.json"

anno_dir = target_dir / "annotations"
images_output_dir = target_dir / "images"

split_names = {
    "train": "instances_train.json",
    "val": "instances_val.json",
    "test": "instances_test.json"
}

for split in ["train", "val", "test"]:
    (images_output_dir / split).mkdir(parents=True, exist_ok=True)
anno_dir.mkdir(parents=True, exist_ok=True)

if all((anno_dir / name).exists() for name in split_names.values()):
    print("Splits and directory structure already exist.")
    exit()

with open(master_json, 'r') as f:
    data = json.load(f)

images = data['images']
annotations = data['annotations']
categories = data['categories']

buckets = [img.get('bucket', 'unknown') for img in images]

train_imgs, temp_imgs, train_buckets, temp_buckets = train_test_split(
    images, buckets, test_size=0.30, stratify=buckets, random_state=seed
)

val_imgs, test_imgs = train_test_split(
    temp_imgs, test_size=0.50, stratify=temp_buckets, random_state=seed
)

def process_split(split_imgs, split_type):
    split_ids = {img['id'] for img in split_imgs}
    
    split_annos = [ann for ann in annotations if ann['image_id'] in split_ids]
    
    output_data = {
        "images": split_imgs,
        "annotations": split_annos,
        "categories": categories
    }
    json_path = anno_dir / split_names[split_type]
    with open(json_path, 'w') as f:
        json.dump(output_data, f, indent=4)

    print(f"Moving {split_type} images...")
    for img in split_imgs:
        file_name = img['file_name']
        src = source_images_dir / file_name
        dst = images_output_dir / split_type / file_name
        
        if src.exists():
            shutil.move(src, dst)
        else:
            print(f"Warning: File {src} not found.")

process_split(train_imgs, "train")
process_split(val_imgs, "val")
process_split(test_imgs, "test")

print("--- Organization Complete ---")
print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)} | Test: {len(test_imgs)}")

In [ ]:
%%writefile deimv2_finetune.yml

__include__: [
  'deimv2_dinov3_l_coco.yml',
]

output_dir: ./outputs

num_classes: 1 
remap_mscoco_category: False

epoches: 32          # 20 for training + EMA for 4n = 12
flat_epoch: 14       # 4 + 20 // 2
no_aug_epoch: 12     # 4n

train_dataloader:
  total_batch_size: 24
  num_workers: 2   
  dataset:
    img_folder: dataset/images/train
    ann_file: dataset/annotations/instances_train.json
    transforms:
      ops:
        - {type: Mosaic, output_size: 320, rotation_range: 0, translation_range: [0.05, 0.05], scaling_range: [0.8, 1.2], probability: 1.0, fill_value: 0, use_cache: True, max_cached_images: 50, random_pop: True}
        - {type: RandomPhotometricDistort, p: 0.2} 
        - {type: RandomZoomOut, fill: 0}
        - {type: RandomIoUCrop, p: 0.5}            
        - {type: SanitizeBoundingBoxes, min_size: 1}
        - {type: RandomHorizontalFlip}             
        - {type: Resize, size: [640, 640]}
        - {type: SanitizeBoundingBoxes, min_size: 1}
        - {type: ConvertPILImage, dtype: 'float32', scale: True}
        - {type: Normalize, mean: [0.485, 0.456, 0.406], std: [0.229, 0.224, 0.225]}
        - {type: ConvertBoxes, fmt: 'cxcywh', normalize: True}
      policy:
        epoch: [4, 14, 20] 

  collate_fn:
    mixup_epochs: [4, 14]       
    stop_epoch: 20              
    copyblend_epochs: [4, 20]   

val_dataloader:
  total_batch_size: 24
  num_workers: 2
  dataset:
    img_folder: dataset/images/val
    ann_file: dataset/annotations/instances_val.json

DEIMCriterion:
  matcher:
    matcher_change_epoch: 18    

In [ ]:
%cp deimv2_finetune.yml DEIMv2/configs/deimv2/deimv2_finetune.yml

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True NCCL_P2P_DISABLE=1 \
    NCCL_IB_DISABLE=1 CUDA_VISIBLE_DEVICES=0,1 torchrun --nproc_per_node=2 \
    --master_port=7777 DEIMv2/train.py \
    -c DEIMv2/configs/deimv2/deimv2_finetune.yml \
    --tuning deimv2.pth \
    --use-amp --seed=0

In [ ]:
!python DEIMv2/train.py \
  -c DEIMv2/configs/deimv2/deimv2_finetune.yml \
  -t outputs/best_stg2.pth \
  --test-only \
  -u val_dataloader.dataset.ann_file=dataset/annotations/instances_test.json val_dataloader.dataset.img_folder=dataset/images/test \
  > outputs/test_results.txt

In [ ]:
!mv outputs/best_stg2.pth outputs/deimv2_anime.pth

In [ ]:
!sed -i 's/torch.rand(32,/torch.rand(1,/g' DEIMv2/tools/deployment/export_onnx.py

In [ ]:
!python DEIMv2/tools/deployment/export_onnx.py --check -c DEIMv2/configs/deimv2/deimv2_finetune.yml -r outputs/deimv2_anime.pth

In [ ]:
import os
import shutil

FILES_TO_KEEP = [
    "outputs/deimv2_anime.onnx",
    "outputs/deimv2_anime.pth",
    "outputs/test_results.txt",
    "outputs/log.txt"
]

BASE_DIR = os.getcwd()

def clean_dir():
    saved_buffers = {}

    for rel_path in FILES_TO_KEEP:
        full_path = os.path.join(BASE_DIR, rel_path)
        if os.path.exists(full_path) and os.path.isfile(full_path):
            try:
                with open(full_path, 'rb') as f:
                    saved_buffers[os.path.basename(rel_path)] = f.read()
                print(f"Staged for preservation: {rel_path}")
            except Exception as e:
                print(f"Failed to read {rel_path}: {e}")
        else:
            print(f"Warning: Target file not found: {rel_path}")

    for item in os.listdir(BASE_DIR):
        item_path = os.path.join(BASE_DIR, item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.remove(item_path)
                print(f"Deleted file: {item_path}")
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
                print(f"Deleted folder: {item_path}")
        except Exception as e:
            print(f"Failed to delete {item_path}: {e}")

    print("\nRestoring preserved files to root...")
    for filename, data in saved_buffers.items():
        destination_path = os.path.join(BASE_DIR, filename)
        try:
            with open(destination_path, 'wb') as f:
                f.write(data)
            print(f"Restored (flattened): {destination_path}")
        except Exception as e:
            print(f"Failed to restore {filename}: {e}")

if __name__ == "__main__":
    clean_dir()